# ByteTrack Evaluation — Adaptive Road Vision

Evaluate **ByteTrack** using the detector selected by Notebook 21 (final detector selection).

### Evaluation scope
- Uses the stable selected detector checkpoint.
- Uses the BDD100K test split only.
- Orders frames only when sequence/frame identifiers are available; image filenames alone may not encode true video continuity.
- Runs Ultralytics ByteTrack with persistent track IDs.
- Reports track-level operational statistics such as active tracks, detections per frame, track lifespan, and ID continuity.
- Evaluates both original and condition-enhanced frames when enhancement checkpoints are available.

### Important limitation
The processed BDD100K metadata used by this project contains object detection annotations, but this notebook does **not** assume that per-object ground-truth tracking IDs exist. Therefore, standard MOT metrics such as MOTA, IDF1, and HOTA are **not claimed** here unless valid tracking ground truth is separately available.

The metrics below are therefore **tracking-behavior diagnostics**, not standard ground-truth tracking accuracy metrics.


In [ ]:
from pathlib import Path
import os
import json
import re
import math
import numpy as np
import pandas as pd
from PIL import Image
import torch
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "datasets").exists() and (candidate / "models").exists():
            return candidate
    return Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

PROJECT_ROOT = find_project_root()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Update PROJECT_ROOT to your actual project folder."
    )
os.chdir(PROJECT_ROOT)

def resolve_dataset_root():
    candidates = [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]
    for cand in candidates:
        if (cand / "metadata" / "final_dataset_metadata.csv").exists():
            return cand
    return candidates[0]

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
METADATA_CANDIDATES = [
    DATASET_ROOT / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2" / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
METADATA_PATH = next((p for p in METADATA_CANDIDATES if p.exists()), None)
SELECTION_PATH = PROJECT_ROOT / "results" / "detection" / "final_detector_selection.json"
RESULT_ROOT = PROJECT_ROOT / "results" / "tracking"
MODEL_ROOT = PROJECT_ROOT / "models"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Metadata:", METADATA_PATH)
print("Ultralytics Device:", DEVICE)
print("PyTorch Device:", TORCH_DEVICE)

## 1. Load the selected detector and test metadata

In [ ]:
from ultralytics import YOLO

if METADATA_PATH is None:
    raise FileNotFoundError(
        "Could not find final dataset metadata. Expected metadata/final_dataset_metadata.csv "
        "under one of the configured dataset folders."
    )

if not SELECTION_PATH.exists():
    fallback = MODEL_ROOT / "detection" / "final_selected_detector.pt"
    if not fallback.exists():
        raise FileNotFoundError(
            f"Final detector selection report is missing ({SELECTION_PATH}) and fallback checkpoint "
            f"does not exist ({fallback}). Run Notebook 21 and Notebook 20 first."
        )
    selected_detector = "Final selected detector (fallback path)"
    selected_model = "selected detector"
    selected_checkpoint = fallback
else:
    selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
    selected_detector = selection.get("selected_detector", "Selected detector")
    selected_model = selection.get("selected_model", "Unknown")
    checkpoint_value = selection.get("stable_downstream_checkpoint") or selection.get("selected_checkpoint")
    if not checkpoint_value:
        raise KeyError(
            "Selection JSON has no stable_downstream_checkpoint or selected_checkpoint. "
            "Re-run Notebook 21 to generate a valid selection report."
        )
    selected_checkpoint = Path(checkpoint_value)
    if not selected_checkpoint.is_absolute():
        selected_checkpoint = (PROJECT_ROOT / selected_checkpoint).resolve()

if not selected_checkpoint.exists():
    raise FileNotFoundError(
        f"Selected detector checkpoint not found: {selected_checkpoint}. "
        "Do not substitute a different model silently; rerun Notebook 21 or correct its report."
    )

print("Selected detector:", selected_detector)
print("Selected model:", selected_model)
print("Checkpoint:", selected_checkpoint)

metadata = pd.read_csv(METADATA_PATH)
if "split" not in metadata.columns:
    raise KeyError(f"Metadata file has no 'split' column: {METADATA_PATH}")
if "project_condition" not in metadata.columns:
    if "condition" in metadata.columns:
        metadata["project_condition"] = metadata["condition"]
    elif "weather" in metadata.columns:
        metadata["project_condition"] = metadata["weather"]
    else:
        metadata["project_condition"] = "Unknown"
if "image_name" not in metadata.columns and "image_path" not in metadata.columns:
    raise KeyError("Metadata must contain image_name or image_path.")

test_meta = metadata[metadata["split"].astype(str).str.strip().str.lower().isin(["test", "testing"])].copy()
if test_meta.empty:
    raise ValueError("No test rows found in metadata. Check split names and dataset preparation.")
print("Test rows in metadata:", len(test_meta))

In [ ]:
from ultralytics import YOLO

try:
    import ultralytics
    print("Ultralytics version:", ultralytics.__version__)
except Exception:
    pass

detector = YOLO(str(selected_checkpoint))
print("Loaded detector:", selected_checkpoint)

## 2. Resolve test-image paths and sequence IDs

BDD-style filenames commonly contain a video/sequence identifier and frame number. This notebook attempts to recover that structure without assuming one exact filename format.


In [ ]:
def resolve_image(row):
    candidates = []
    for key in ["image_path", "image_name"]:
        if key not in row.index or pd.isna(row[key]):
            continue
        value = str(row[key]).strip()
        if not value:
            continue
        p = Path(value)
        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.extend([
                PROJECT_ROOT / p,
                DATASET_ROOT / p,
                DATASET_ROOT / "images" / "test" / p.name,
                DATASET_ROOT / "images" / "testing" / p.name,
            ])
    name = str(row.get("image_name", ""))
    if name:
        basename = Path(name).name
        for folder in ["test", "testing"]:
            for ext in ["", ".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                candidate_name = basename if ext == "" else Path(basename).stem + ext
                candidates.append(DATASET_ROOT / "images" / folder / candidate_name)
    seen = set()
    for p in candidates:
        key = str(p)
        if key not in seen and p.is_file():
            return p.resolve()
        seen.add(key)
    return None


def parse_sequence_frame(row):
    """Return sequence/frame identity and whether sequence membership is explicit.

    Filename-derived grouping is only a heuristic. It is useful for exploratory ordering,
    but it is NOT treated as verified temporal ground truth for continuity claims.
    """
    for seq_key in ["sequence_id", "video_id", "video_name", "sequence"]:
        if seq_key in row.index and pd.notna(row[seq_key]) and str(row[seq_key]).strip():
            sequence_id = str(row[seq_key]).strip()
            frame_no = 0
            for frame_key in ["frame_number", "frame_id", "frame_index"]:
                if frame_key in row.index and pd.notna(row[frame_key]):
                    match = re.search(r"(\d+)", str(row[frame_key]))
                    if match:
                        frame_no = int(match.group(1))
                        break
            return sequence_id, frame_no, True, "explicit_metadata"

    name = Path(str(row.get("image_name", row.get("image_path", "unknown")))).stem
    match = re.match(r"(.+?)[-_](\d+)$", name)
    if match:
        return match.group(1), int(match.group(2)), False, "filename_heuristic_only"
    return name, 0, False, "single_image_no_sequence_id"


test_meta["resolved_image_path"] = test_meta.apply(resolve_image, axis=1)
missing_count = int(test_meta["resolved_image_path"].isna().sum())
if missing_count:
    print(f"Warning: {missing_count} test metadata rows have no resolvable image and will be skipped.")
test_meta = test_meta[test_meta["resolved_image_path"].notna()].copy()
if test_meta.empty:
    raise FileNotFoundError("No test images could be resolved. Check image_path and dataset folders.")

seq_info = test_meta.apply(parse_sequence_frame, axis=1, result_type="expand")
seq_info.columns = ["sequence_id", "frame_number", "sequence_evidence", "sequence_source"]
test_meta[["sequence_id", "frame_number", "sequence_evidence", "sequence_source"]] = seq_info
if not test_meta["sequence_evidence"].all():
    print("WARNING: Some rows lack explicit video/sequence metadata. Continuity values for those groups will be marked unavailable, not zero.")
test_meta = test_meta.sort_values(["sequence_id", "frame_number", "image_name" if "image_name" in test_meta.columns else "resolved_image_path"]).reset_index(drop=True)
print("Resolvable test images:", len(test_meta))
print("Sequence groups:", test_meta["sequence_id"].nunique())
print("Rows with explicit sequence metadata:", int(test_meta["sequence_evidence"].sum()))
display(test_meta[[c for c in ["image_name", "sequence_id", "frame_number", "project_condition", "sequence_evidence", "sequence_source"] if c in test_meta.columns]].head(20))


## 3. ByteTrack configuration

In [ ]:
TRACKER_CONFIG = "bytetrack.yaml"
IMAGE_SIZE = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

# Set this to an integer for a quick smoke test (e.g., 50 or 100).
# Set to None to evaluate the full available test set.
MAX_FRAMES = None

print("Tracker:", TRACKER_CONFIG)
print("Image size:", IMAGE_SIZE)
print("Confidence threshold:", CONF_THRESHOLD)
print("IoU threshold:", IOU_THRESHOLD)
print("Max frames:", MAX_FRAMES if MAX_FRAMES is not None else "ALL")

## 4. Optional enhancement models

The enhancement architecture is reconstructed only for inference. If a condition-specific checkpoint is missing, that condition is evaluated using the original frame rather than silently inventing an enhanced result.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x):
        return x + self.block(x)


class ConditionEnhancer(nn.Module):
    """
    Compact residual encoder-decoder matching the architecture
    trained in Notebooks 14 (Night), 15 (Rain), and 16 (Fog).
    """
    def __init__(self, base_channels=32, num_res_blocks=4):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.res = nn.Sequential(*[ResidualBlock(base_channels * 4) for _ in range(num_res_blocks)])
        self.dec2 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.out = nn.Conv2d(base_channels, 3, 3, padding=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        z = self.res(e3)

        z = F.interpolate(z, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        z = self.dec2(z) + e2

        z = F.interpolate(z, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        z = self.dec1(z) + e1

        residual = torch.tanh(self.out(z)) * 0.5
        return torch.clamp(x + residual, 0.0, 1.0)


# Backward compatibility alias
EnhancementNet = ConditionEnhancer


def load_enhancer(path):
    checkpoint = torch.load(path, map_location=TORCH_DEVICE)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict):
        state = checkpoint
    else:
        state = checkpoint.state_dict()

    cleaned = {k.replace("module.", ""): v for k, v in state.items()}
    model = ConditionEnhancer().to(TORCH_DEVICE)
    try:
        model.load_state_dict(cleaned, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            f"Checkpoint architecture does not match ConditionEnhancer: {path}."
        ) from exc
    model.eval()
    return model


enhancer_paths = {
    "Night": MODEL_ROOT / "enhancement" / "night" / "night_enhancer_best.pth",
    "Rain": MODEL_ROOT / "enhancement" / "rain" / "rain_enhancer_best.pth",
    "Fog": MODEL_ROOT / "enhancement" / "fog" / "fog_enhancer_best.pth",
}

enhancers = {}

for condition, path in enhancer_paths.items():
    if path.exists():
        try:
            enhancers[condition] = load_enhancer(path)
            print("Loaded:", condition, path)
        except Exception as e:
            print("Could not load", condition, ":", e)
    else:
        print("Missing:", path)

print("Active enhancement models loaded:", list(enhancers.keys()))

enhance_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])


def enhance_image(image, condition):
    if condition not in enhancers:
        return image.copy()

    tensor = enhance_transform(image).unsqueeze(0).to(TORCH_DEVICE)

    with torch.inference_mode():
        output = enhancers[condition](tensor)[0].cpu()

    enhanced = transforms.ToPILImage()(output)
    return enhanced.resize(image.size, Image.Resampling.BILINEAR)

## 5. Tracking helper

Each sequence gets a fresh ByteTrack state. `persist=True` keeps track IDs across consecutive frames in that sequence.


In [ ]:
def reset_tracker(model):
    """Safely reset ByteTrack state between video sequences without reloading weights from disk."""
    if hasattr(model, "predictor") and model.predictor is not None:
        if hasattr(model.predictor, "trackers"):
            for t in model.predictor.trackers:
                t.reset()
                if hasattr(t, "reset_id"):
                    t.reset_id()


def make_tracker():
    return detector


def run_sequence(sequence_df, mode="original", tracker=None):
    if tracker is None:
        tracker = detector
    reset_tracker(tracker)
    rows = []

    for i, row in enumerate(sequence_df.itertuples(index=False)):
        image_path = Path(row.resolved_image_path)
        condition = str(row.project_condition)

        image = Image.open(image_path).convert("RGB")

        if mode == "enhanced":
            input_image = enhance_image(image, condition)
        else:
            input_image = image

        results = tracker.track(
            source=np.asarray(input_image),
            persist=True,
            tracker=TRACKER_CONFIG,
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            device=DEVICE,
            verbose=False,
        )

        result = results[0]

        if result.boxes is None or len(result.boxes) == 0:
            detection_count = 0
            track_ids = np.array([], dtype=np.int64)
            confs = np.array([], dtype=np.float32)
            classes = np.array([], dtype=np.int64)
            untracked_detections = 0
        else:
            detection_count = int(len(result.boxes))
            ids = result.boxes.id
            if ids is None:
                track_ids = np.array([], dtype=np.int64)
                untracked_detections = detection_count
            else:
                track_ids = ids.cpu().numpy().astype(np.int64)
                untracked_detections = max(0, detection_count - len(track_ids))
            confs = result.boxes.conf.cpu().numpy().astype(np.float32)
            classes = result.boxes.cls.cpu().numpy().astype(np.int64)

        unique_ids = set(track_ids.tolist())

        rows.append({
            "sequence_id": str(row.sequence_id),
            "sequence_evidence": bool(getattr(row, "sequence_evidence", False)),
            "sequence_source": str(getattr(row, "sequence_source", "unknown")),
            "image_name": str(getattr(row, "image_name", Path(row.resolved_image_path).name)),
            "frame_number": int(row.frame_number),
            "condition": condition,
            "mode": mode,
            "detections": detection_count,
            "tracked_detections": int(len(track_ids)),
            "untracked_detections": int(untracked_detections),
            "active_tracks": int(len(unique_ids)),
            "mean_confidence": float(confs.mean()) if len(confs) else 0.0,
            "track_ids": ",".join(map(str, sorted(unique_ids))),
            "class_ids": ",".join(map(str, classes.tolist())),
        })

    return rows

## 6. Run ByteTrack on the test sequences

For a full experiment this can be computationally heavy because each frame is passed through the detector. Use `MAX_FRAMES` for a smoke test first if needed.


In [ ]:
eval_meta = test_meta.copy()

if MAX_FRAMES is not None:
    eval_meta = eval_meta.head(MAX_FRAMES).copy()

all_rows = []

for mode in ["original", "enhanced"]:
    print(f"Running ByteTrack mode: {mode}")

    for sequence_id, seq_df in eval_meta.groupby("sequence_id", sort=False):
        # Require chronological order inside each sequence.
        seq_df = seq_df.sort_values(
            ["frame_number", "image_name"]
        )

        try:
            rows = run_sequence(seq_df, mode=mode, tracker=detector)
            all_rows.extend(rows)
        except Exception as e:
            print(f"Sequence failed: {sequence_id} | {e}")

    print("Completed:", mode)

tracking_df = pd.DataFrame(all_rows)

if tracking_df.empty:
    raise RuntimeError("No tracking results were produced.")

print("Tracking rows:", len(tracking_df))
display(tracking_df.head())

## 7. Track continuity diagnostics

Because tracking ground-truth IDs are not available in the project's current detection metadata, these metrics describe tracker behavior rather than standard MOT accuracy.


In [ ]:
def sequence_metrics(df):
    rows = []

    for (mode, sequence_id), g in df.groupby(
        ["mode", "sequence_id"], sort=False
    ):
        active_counts = g["active_tracks"].to_numpy()
        detection_counts = g["detections"].to_numpy()

        # Number of track IDs observed over the sequence.
        all_ids = set()
        for text in g["track_ids"].fillna(""):
            if text:
                all_ids.update(
                    int(x) for x in str(text).split(",") if x.strip()
                )

        # Track appearances per frame.
        appearance_counts = {}
        for text in g["track_ids"].fillna(""):
            if text:
                for x in str(text).split(","):
                    if x.strip():
                        tid = int(x)
                        appearance_counts[tid] = appearance_counts.get(tid, 0) + 1

        lifespans = list(appearance_counts.values())

        rows.append({
            "mode": mode,
            "sequence_id": sequence_id,
            "frames": len(g),
            "mean_detections_per_frame": float(detection_counts.mean()) if len(detection_counts) else 0.0,
            "mean_active_tracks": float(active_counts.mean()) if len(active_counts) else 0.0,
            "max_active_tracks": int(active_counts.max()) if len(active_counts) else 0,
            "unique_track_ids": len(all_ids),
            "mean_track_lifespan_frames": (
                float(np.mean(lifespans)) if lifespans else 0.0
            ),
            "max_track_lifespan_frames": (
                int(max(lifespans)) if lifespans else 0
            ),
        })

    return pd.DataFrame(rows)


sequence_df = sequence_metrics(tracking_df)
display(sequence_df.head(20))

In [ ]:
overall_tracking = (
    tracking_df
    .groupby("mode")
    .agg(
        sequences=("sequence_id", "nunique"),
        frames=("image_name", "count"),
        mean_detections_per_frame=("detections", "mean"),
        mean_active_tracks=("active_tracks", "mean"),
        max_active_tracks=("active_tracks", "max"),
        mean_confidence=("mean_confidence", "mean"),
    )
    .reset_index()
)

display(overall_tracking)


## 8. Track ID continuity diagnostics

A simple ID-overlap diagnostic is calculated only when explicit sequence metadata identifies multi-frame sequences. If the metadata contains independent images or only filename-based guesses, continuity is reported as unavailable (NaN) and the chart is skipped rather than displaying a misleading empty/zero plot. This is **not** an IDF1/MOTA score.


In [ ]:
def continuity_metrics(df):
    """Exploratory ID-overlap diagnostic; only calculate for explicit sequence metadata.

    Without frame-level tracking ground truth, this is not MOTA/IDF1/HOTA.
    """
    rows = []
    for (mode, sequence_id), g in df.groupby(["mode", "sequence_id"], sort=False):
        g = g.sort_values(["frame_number", "image_name"])
        # sequence_evidence is carried through from the input metadata when available.
        explicit_sequence = bool(g["sequence_evidence"].astype(bool).all()) if "sequence_evidence" in g.columns else False
        transitions = 0
        continued = 0
        total_track_observations = 0
        previous_ids = None
        for text in g["track_ids"].fillna(""):
            current_ids = set(int(x) for x in str(text).split(",") if x.strip()) if text else set()
            total_track_observations += len(current_ids)
            if previous_ids is not None:
                transitions += 1
                continued += len(previous_ids.intersection(current_ids))
            previous_ids = current_ids

        valid = explicit_sequence and len(g) > 1 and transitions > 0 and total_track_observations > 0
        rows.append({
            "mode": mode,
            "sequence_id": sequence_id,
            "frames": int(len(g)),
            "explicit_sequence_metadata": explicit_sequence,
            "consecutive_frame_transitions": int(transitions),
            "continued_track_observations": int(continued),
            "track_observations": int(total_track_observations),
            "continuity_ratio": float(continued / total_track_observations) if valid else np.nan,
            "continuity_valid": bool(valid),
        })
    return pd.DataFrame(rows)

continuity_df = continuity_metrics(tracking_df)
display(continuity_df.head(20))

valid_continuity = continuity_df[continuity_df["continuity_valid"]].copy()
if valid_continuity.empty:
    continuity_summary = pd.DataFrame(columns=["mode", "sequences", "mean_continuity_ratio", "median_continuity_ratio"])
    print("Continuity diagnostic unavailable: no multi-frame groups with explicit sequence metadata and track observations were found. This is not a 0% score.")
else:
    continuity_summary = (
        valid_continuity.groupby("mode")
        .agg(
            sequences=("sequence_id", "nunique"),
            mean_continuity_ratio=("continuity_ratio", "mean"),
            median_continuity_ratio=("continuity_ratio", "median"),
        )
        .reset_index()
    )

display(continuity_summary)


## 9. Visualize tracker behavior

In [ ]:
if not overall_tracking.empty:
    plt.figure(figsize=(9, 5))
    plt.bar(overall_tracking["mode"], overall_tracking["mean_active_tracks"])
    plt.ylabel("Mean active tracks / frame")
    plt.title("ByteTrack Active-Track Behavior")
    plt.tight_layout()
    plt.show()
else:
    print("No overall tracking metrics available to plot.")

if not continuity_summary.empty and continuity_summary["mean_continuity_ratio"].notna().any():
    plot_data = continuity_summary.dropna(subset=["mean_continuity_ratio"])
    plt.figure(figsize=(9, 5))
    plt.bar(plot_data["mode"], plot_data["mean_continuity_ratio"])
    plt.ylim(0, 1)
    plt.ylabel("Mean track-ID overlap ratio (diagnostic)")
    plt.title("ByteTrack Track Continuity Diagnostic")
    plt.tight_layout()
    plt.show()
else:
    print("Continuity chart skipped: valid temporal sequence data is unavailable. The previous empty chart should not be interpreted as zero continuity.")


## 10. Save tracking results

In [ ]:
per_frame_path = RESULT_ROOT / "bytetrack_per_frame.csv"
sequence_path = RESULT_ROOT / "bytetrack_sequence_metrics.csv"
overall_path = RESULT_ROOT / "bytetrack_overall_metrics.csv"
continuity_path = RESULT_ROOT / "bytetrack_continuity_metrics.csv"
report_path = RESULT_ROOT / "bytetrack_evaluation_report.json"

tracking_df.to_csv(per_frame_path, index=False)
sequence_df.to_csv(sequence_path, index=False)
overall_tracking.to_csv(overall_path, index=False)
continuity_df.to_csv(continuity_path, index=False)

report = {
    "selected_detector": selected_detector,
    "selected_model": selected_model,
    "selected_checkpoint": str(selected_checkpoint),
    "tracker": TRACKER_CONFIG,
    "image_size": IMAGE_SIZE,
    "confidence_threshold": CONF_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "test_images_evaluated": int(eval_meta["resolved_image_path"].nunique()),
    "sequences_evaluated": int(eval_meta["sequence_id"].nunique()),
    "rows_without_verified_sequence_metadata": int((~eval_meta["sequence_evidence"].astype(bool)).sum()),
    "valid_continuity_sequence_groups": int(continuity_df["continuity_valid"].sum()) if not continuity_df.empty else 0,
    "enhancement_models_loaded": sorted(enhancers.keys()),
    "overall_tracking_metrics": overall_tracking.to_dict(orient="records"),
    "continuity_summary": continuity_summary.to_dict(orient="records"),
    "limitation": (
        "No per-object tracking ground-truth IDs are assumed from the current "
        "project metadata. Therefore MOTA, IDF1, HOTA, and ID-switch accuracy "
        "are not reported. Results are operational ByteTrack diagnostics. If true video/sequence IDs "
        "are absent, per-frame track continuity is not a valid temporal tracking evaluation."
    ),
}

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", per_frame_path)
print("Saved:", sequence_path)
print("Saved:", overall_path)
print("Saved:", continuity_path)
print("Saved:", report_path)


## Final check

The saved report provides the evidence needed for the downstream tracking/pipeline stages:

```text
results/tracking/bytetrack_evaluation_report.json
```

For a rigorous MOT benchmark, provide a test set with frame-level ground-truth object IDs and extend this notebook with standard MOTA/IDF1/HOTA evaluation.
